<a href="https://colab.research.google.com/github/tselane2110/NaturalLanguageProcessing/blob/main/Urdu_Sarcasm_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# cloning into Git Repo
!git clone https://github.com/tselane2110/NaturalLanguageProcessing

fatal: destination path 'NaturalLanguageProcessing' already exists and is not an empty directory.


In [2]:
%cd /content/NaturalLanguageProcessing

/content/NaturalLanguageProcessing


In [3]:
%ls

projectGuidelines.pdf  Urdu_Sarcasm_Classification.ipynb
README.md              urdu_sarcastic_dataset.csv


## 1. Data Analysis and Preprocessing

In [4]:
import re
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_colwidth', None)   # show full column width
df = pd.read_csv('urdu_sarcastic_dataset.csv')
df.head()

,urdu_text,is_sarcastic,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2,کامران خان آپکی دن بھریہ زمہ داری لگائی گئی اپوزیشن کی کردار کشی اوراس پربھونکناہےآپ خوشامدگری وچاپلوسی سےاورکتنی دولت کماناچاہتےہیں موٹرسائیکل سےپیجارو پراڈو تک کےسفرمیں ضمیرکی لاش سےاٹھتی بدبوآپ کی ناک بند نہیں کرتی ہے 🙏نوٹ آپ سب سےالتجاگزارش ہےہمیں بھی فالوکریں شکریہ,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,نہیں پائین 😎,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,`` مراد علی شاہ کے بھیس میں ڈی جی آئی ایس آئی تھے '' حامد میر😁,1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns

Index(['urdu_text', 'is_sarcastic', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7'],
      dtype='object')

### i. Not Including Unnecessary Columns

In [6]:
# not using the extra unimportant columns

df = df.iloc[:, :-6]
df.head()

,urdu_text,is_sarcastic
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1.0
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1.0
2,کامران خان آپکی دن بھریہ زمہ داری لگائی گئی اپوزیشن کی کردار کشی اوراس پربھونکناہےآپ خوشامدگری وچاپلوسی سےاورکتنی دولت کماناچاہتےہیں موٹرسائیکل سےپیجارو پراڈو تک کےسفرمیں ضمیرکی لاش سےاٹھتی بدبوآپ کی ناک بند نہیں کرتی ہے 🙏نوٹ آپ سب سےالتجاگزارش ہےہمیں بھی فالوکریں شکریہ,0.0
3,نہیں پائین 😎,0.0
4,`` مراد علی شاہ کے بھیس میں ڈی جی آئی ایس آئی تھے '' حامد میر😁,1.0


In [7]:
# renaming the text columns
df = df.rename(columns={'urdu_text': 'text'})
df.head(2)

,text,is_sarcastic
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1.0
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1.0


### ii. Dropping NaN rows

In [8]:
# finding NaN rows

df.isna().sum()

,0
text,105
is_sarcastic,56


In [9]:
# dropping all NaN rows

df = df.dropna()

In [10]:
df.head(2)

,text,is_sarcastic
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1.0
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1.0


### iii. Converting DataType from Float to Int of `is_sarcastic` Column

In [11]:
# converting the datatype of is_sarcastic from float to int

df['is_sarcastic'] = df['is_sarcastic'].astype(int)
df.head(2)

,text,is_sarcastic
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1


### iv. Isolating All Emojis and Some Punctuation Into Separate Columns
* Removing all emojis and punctuations from all rows of the `text` column

In [12]:
!pip install emoji

In [13]:
import re
import emoji
import pandas as pd

# 1. Function to extract emojis into a new column and remove them from text
def extract_and_remove_emojis(text):
    text = str(text)
    # Find all emojis in the text
    found_emojis = "".join([char for char in text if char in emoji.EMOJI_DATA])
    # Remove emojis from the text
    cleaned_text = "".join(
        [char for char in text if char not in emoji.EMOJI_DATA]
    )
    return cleaned_text, found_emojis


# 2. Function to extract specific punctuations (?, ., !) and remove all punctuation except #
def extract_and_remove_punctuation(text):
    text = str(text)
    # Find all ?, ., ! marks
    found_punc = "".join(re.findall(r"[?.!]", text))

    # Remove all sorts of punctuation from the text
    # string.punctuation covers all standard symbols

    import string

    punct_to_remove = string.punctuation.replace("#", "")

    translator = str.maketrans("", "", punct_to_remove)
    cleaned_text = text.translate(translator)

    return cleaned_text, found_punc


# Apply Emoji extraction
df["text_no_emoji"], df["emojis"] = zip(*df["text"].apply(extract_and_remove_emojis))

# Apply Punctuation extraction on the emoji-cleaned text (or original text)
df["cleaned_text"], df["punctuations"] = zip(
    *df["text_no_emoji"].apply(extract_and_remove_punctuation)
)

# Clean up intermediate column if you want
df = df.drop(columns=["text_no_emoji"])

In [14]:
df.head(10)

,text,is_sarcastic,emojis,cleaned_text,punctuations
0,🤣😂😂 ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے 😐😐😐🤣,1,🤣😂😂😐😐😐🤣,ہو لینے دے میری شادی فسادن ٹھیک ہے کوجی نہیں چاہیے,
1,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں😂😂,1,😂😂,چل مہمانوں میں کھانا سرو کر چڑیل چاچی نوں دسدی آں میں,
2,کامران خان آپکی دن بھریہ زمہ داری لگائی گئی اپوزیشن کی کردار کشی اوراس پربھونکناہےآپ خوشامدگری وچاپلوسی سےاورکتنی دولت کماناچاہتےہیں موٹرسائیکل سےپیجارو پراڈو تک کےسفرمیں ضمیرکی لاش سےاٹھتی بدبوآپ کی ناک بند نہیں کرتی ہے 🙏نوٹ آپ سب سےالتجاگزارش ہےہمیں بھی فالوکریں شکریہ,0,🙏,کامران خان آپکی دن بھریہ زمہ داری لگائی گئی اپوزیشن کی کردار کشی اوراس پربھونکناہےآپ خوشامدگری وچاپلوسی سےاورکتنی دولت کماناچاہتےہیں موٹرسائیکل سےپیجارو پراڈو تک کےسفرمیں ضمیرکی لاش سےاٹھتی بدبوآپ کی ناک بند نہیں کرتی ہے نوٹ آپ سب سےالتجاگزارش ہےہمیں بھی فالوکریں شکریہ,
3,نہیں پائین 😎,0,😎,نہیں پائین,
4,`` مراد علی شاہ کے بھیس میں ڈی جی آئی ایس آئی تھے '' حامد میر😁,1,😁,مراد علی شاہ کے بھیس میں ڈی جی آئی ایس آئی تھے حامد میر,
5,قابل اعتبار ہی اکثر قاتل اعتبار ہوتے ہیں 💔🔥,1,💔🔥,قابل اعتبار ہی اکثر قاتل اعتبار ہوتے ہیں,
6,انساں کو تھکا دیتا ہے سوچوں کا سفر بھی ... 🍁🥀,0,🍁🥀,انساں کو تھکا دیتا ہے سوچوں کا سفر بھی,...
7,حامد میر صاحب ویلڈن👏😊,0,👏😊,حامد میر صاحب ویلڈن,
8,یار وچارہ ویلا ہوندا ہے اس آرے لگا ہویا ہے😂😂 تسی تے پکے نجومی ہو اسی منندے ہاں,1,😂😂,یار وچارہ ویلا ہوندا ہے اس آرے لگا ہویا ہے تسی تے پکے نجومی ہو اسی منندے ہاں,
9,یہ سمجھتے ہیں سارا پاکستان بیوقوف ھے 😂😂😂,1,😂😂😂,یہ سمجھتے ہیں سارا پاکستان بیوقوف ھے,


### v. Dropping and renaming Columns

In [15]:
# Dropping the uncleaned `text` column
df.drop(columns='text', inplace=True)

In [16]:
# Renaming `cleaned_text` to `text`
df = df.rename(columns={'cleaned_text': 'text'})

In [17]:
df.columns

Index(['is_sarcastic', 'emojis', 'text', 'punctuations'], dtype='object')

### vi. Finding rows with Hashtags

In [18]:
# Filter rows where the 'text' column contains a hashtag
hashtag_rows = df[df["text"].str.contains("#", na=False)]

# See how many rows have hashtags
print(f"Total rows with hashtags: {len(hashtag_rows)}")

# Display the first few rows (or all of them)
print(hashtag_rows[["text"]].head(10))

Total rows with hashtags: 756
                                                                                                                                                                                                                                text
5277                                  SHABAZGIL جب کتے زیادہ بھونکنے لگے تو ان کے آگے رکھنے کیلئے ہڈیوں کی تعداد بڑھا دواقوال کرائم منسٹر شوباز شریف برادر گوالمنڈیلا لندن والے سے اقتباس#امپورٹڈحکومتنامنظور #امپورٹڈعدالتیںنامنظور
5475  “Point to be Noted”ویسے یہ کیسے پتہ چلا کہ میر جعفر اور میر صادق غدار تھے ؟ ایسٹ انڈیا کمپنی کا کوئئ ایسا لیٹر موجود ہے جس میں  سازش کا ذکر ہو ؟ یا کوئی ایسا پلان لکھا گیا ہو ۔منقول #امپورٹڈحکومتنامنظور#امپوڑٹڈحکومتنامنظور
5479                                                   FarooquiJameel چین میں کرپشن پر سزائے موت دی جاتی ہے اور ھمارے ملک میں کرپشن پر وزارتیں دی جاتیں ہیں جتنی بڑی کرپشن اتنی بڑی  وزارت اور عہدہ دیا جاتا ہے #امپوڑٹڈحکومتنامنظور
5611                                                  